In [7]:
import math
import os.path


def highlight_row_max(s):
    if s.name == "Dataset":
        return [""] * len(s)
    if s.name == "Avg Rank":
        is_min = s == s.min()
        return ["background-color: green; color: white" if v else "" for v in is_min]
    is_max = s == s.max()
    return ["background-color: green; color: white" if v else "" for v in is_max]


def highlight_row_min(s):
    if s.name == "Dataset":
        return [""] * len(s)
    if s.name == "Wins":
        is_max = s == s.max()
        return ["background-color: green; color: white" if v else "" for v in is_max]
    is_min = s == s.min()
    return ["background-color: green; color: white" if v else "" for v in is_min]


def sort_dataset(df):
    desired_order = [
        "fingerprint_high_freq",
        "fingerprint_mid_freq",
        "fingerprint_low_freq",
        "fingerprint_cardiac",
        "fingerprint_ten",
        "fingerprint_ten_mid",
        "CPI_high_freq",
        "CPI_mid_freq",
        "CPI_low_freq",
        "CPI_cardiac",
        "CPI_ten",
        "CPI_ten_mid",
        "CPI+fingerprint_high_freq",
        "CPI+fingerprint_mid_freq",
        "CPI+fingerprint_low_freq",
        "CPI+fingerprint_cardiac",
        "CPI+fingerprint_ten",
        "CPI+fingerprint_ten_mid",
        "Avg Rank",
        "Wins"
    ]
    order_map = {name: i for i, name in enumerate(desired_order)}
    df["__order"] = df["Dataset"].map(order_map)
    df_sorted = df.sort_values("__order").drop(columns="__order")
    return df_sorted

In [8]:
import pandas as pd
import numpy as np

XofN_none_results = pd.read_csv("../XofN_none/table_results.csv", index_col=0)
XofN_none_results["XofN method"] = "XofN_none"
XofN_none_results["include_original_features"] = "N/A"
XofN_none_results["XofN time (s); PCT tr. time (s)"] = XofN_none_results["PCT tr. time (s)"]
XofN_none_results = XofN_none_results.drop(columns=["PCT tr. time (s)"])

# XofN_naive_results = pd.read_csv("XofN_naive/table_results.csv", index_col=0)
# XofN_naive_results["XofN method"] = "XofN_naive"

XofN_jaccard_max_results = pd.read_csv("../XofN_jaccard/table_results.csv", index_col=0)
XofN_jaccard_max_results["XofN method"] = "XofN Jac-max"

XofN_filter_results = pd.read_csv("../XofN_filter/table_results.csv", index_col=0)
XofN_filter_results["XofN method"] = "XofN VarRed"

# XofN_filter_jaccard_results = pd.read_csv("XofN_filter_jaccard/table_results.csv", index_col=0)
# XofN_filter_jaccard_results["XofN method"] = "XofN_filter_jaccard"

# XofN_jaccard_min_results = pd.read_csv("XofN_jaccard_min/table_results.csv", index_col=0)
# XofN_jaccard_min_results["XofN method"] = "XofN_jaccard_min"

# XofN_wrapper_results = pd.read_csv("XofN_wrapper/table_results.csv")
# XofN_wrapper_results["XofN method"] = "XofN_wrapper"

XofN_Vouk_results = pd.read_csv("../../single_label_approach/vouk/table_results.csv")
XofN_Vouk_results["XofN method"] = "XofN Vouk"

In [9]:
all_cols = set(XofN_none_results.columns) | set(XofN_filter_results.columns)
df1 = XofN_none_results.reindex(columns=all_cols)
# df2 = XofN_naive_results.reindex(columns=all_cols)
df3 = XofN_jaccard_max_results.reindex(columns=all_cols)
df4 = XofN_filter_results.reindex(columns=all_cols)
# df5 = XofN_filter_jaccard_results.reindex(columns=all_cols)

# df5 = XofN_wrapper_results.reindex(columns=all_cols)
# df6 = XofN_jaccard_min_results.reindex(columns=all_cols)
df7 = XofN_Vouk_results.reindex(columns=all_cols)

combined_df = pd.concat([df1, df3, df4, df7], ignore_index=True)

column_map = {
    "dataset": "Dataset",
    "XofN method": "XofN method",
    "include_original_features": "Ung or Org",
    'averageAUROC': 'Macro-AUROC',
    'HammingLoss': 'Hamming Loss',
    'SubsetAccuracy': 'Subset Accuracy',
    'MacroFOne': 'Macro-F1',
    'MacroRecall': 'Macro-Recall',
    'MacroPrecision': 'Macro-Precision',
    'RankingLoss': 'Ranking Loss',
    'Nodes; Leaves': 'Nodes; Leaves',
    '#XofN; #Feat/XofN': '#XofN; #Feat/XofN',
    '# Ung. Feats': '#Ung. Feats',
    'XofN time (s); PCT tr. time (s)': 'XofN time (s); PCT tr. time (s)'
}
column_order = [column_map[col] for col in column_map]
combined_df = combined_df.rename(columns=column_map)
combined_df = combined_df.reindex(columns=column_order)
combined_df["Ung or Org"] = combined_df["Ung or Org"].map({
    "no_org": "Ung",
    "with_org": "Org",
    "N/A": "N/A"
})
combined_df['#XofN; #Feat/XofN'] = combined_df['#XofN; #Feat/XofN'].replace(np.nan, "N/A")
combined_df['#Ung. Feats'] = combined_df['#Ung. Feats'].replace(np.nan, "N/A")
combined_df.to_csv("all_results_table.csv", index=False)
print(combined_df.to_string())

                       Dataset   XofN method Ung or Org  Macro-AUROC  Hamming Loss  Subset Accuracy  Macro-F1  Macro-Recall  Macro-Precision  Ranking Loss Nodes; Leaves #XofN; #Feat/XofN #Ung. Feats XofN time (s); PCT tr. time (s)
0         fingerprint_mid_freq     XofN_none        N/A        0.574         0.386            0.231     0.415         0.373            0.474         0.258   161.0; 81.0               N/A         N/A                             1.1
1         fingerprint_low_freq     XofN_none        N/A        0.511         0.246            0.371     0.020         0.017            0.077         0.302      8.8; 4.9               N/A         N/A                             1.1
2        fingerprint_high_freq     XofN_none        N/A        0.517         0.227            0.519     0.870         0.983            0.782         0.164     18.8; 9.9               N/A         N/A                             1.0
3          fingerprint_cardiac     XofN_none        N/A        0.517        

In [10]:
approaches_list = {
    "all": ["Dataset", "XofN_none N/A",
            "XofN Jac-max Ung", "XofN Jac-max Org",
            "XofN VarRed Ung", "XofN VarRed Org",
            "XofN Vouk Ung", "XofN Vouk Org",
            # "XofN_filter_jaccard Ung", "XofN_filter_jaccard Org",
            # "XofN_naive Ung", "XofN_naive Org",
            # "XofN_jaccard_min Ung", "XofN_jaccard_min Org",
            # "XofN_wrapper Ung", "XofN_wrapper Org"
            ],
    "all Ung": ["Dataset", "XofN_none N/A", "XofN Jac-max Ung", "XofN VarRed Ung", "XofN Vouk Ung"],
    "all Org": ["Dataset", "XofN_none N/A", "XofN Jac-max Org", "XofN VarRed Org", "XofN Vouk Org"],
    "Jac-max": ["Dataset", "XofN_none N/A", "XofN Jac-max Ung", "XofN Jac-max Org", "XofN Vouk Ung",
                "XofN Vouk Org"],
    "Jac-max Ung": ["Dataset", "XofN_none N/A", "XofN Jac-max Ung", "XofN Vouk Ung"],
    "Jac-max Org": ["Dataset", "XofN_none N/A", "XofN Jac-max Org", "XofN Vouk Org"],
    "VarRed": ["Dataset", "XofN_none N/A", "XofN VarRed Ung", "XofN VarRed Org", "XofN Vouk Ung", "XofN Vouk Org"],
    "VarRed Ung": ["Dataset", "XofN_none N/A", "XofN VarRed Ung", "XofN Vouk Ung"],
    "VarRed Org": ["Dataset", "XofN_none N/A", "XofN VarRed Org", "XofN Vouk Org"],
}

minimise_measures = ["Hamming Loss", "Ranking Loss"]

In [34]:
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp
import shutil
from itertools import combinations
from scipy import stats


def save_measure_table(measure, comp_name):
    minimise = measure in minimise_measures
    highlight = highlight_row_min if minimise else highlight_row_max
    ascending = True if minimise else False
    path = f"{measure}/{comp_name}/{measure}"
    if os.path.exists(f"{measure}/{comp_name}"):
        shutil.rmtree(f"{measure}/{comp_name}")
    os.mkdir(f"{measure}/{comp_name}")
    all_results_table = pd.read_csv("all_results_table.csv")
    all_results_table.loc[
        all_results_table["XofN method"] == "XofN_none", "Ung or Org"
    ] = "N/A"
    all_results_table.columns = all_results_table.columns.str.strip()
    pivot = all_results_table.pivot_table(
        index=["Dataset"],
        columns=["XofN method", "Ung or Org"],
        values=measure,
        aggfunc="first"
    )
    pivot.columns = [f"{a} {b}" for a, b in pivot.columns]
    pivot = pivot.reset_index()
    pivot = pivot.sort_values(by="Dataset", ascending=False)
    pivot_column_order = approaches.copy()
    pivot = pivot.reindex(columns=pivot_column_order)
    pivot = pivot.replace("nan", np.nan)
    numeric = pivot.select_dtypes(include=["number"])
    ranks = numeric.rank(
        axis=1,
        ascending=ascending,
        method="average",
        na_option="bottom"
    )
    avg_ranks = ranks.mean(axis=0).round(2).to_frame().T
    print("\n--", measure, comp_name, "--")
    avg_ranks["Dataset"] = "Avg Rank"
    avg_ranks.index = ["Avg Rank"]
    pivot = pd.concat([pivot, avg_ranks], ignore_index=False)
    cols_to_highlight = [col for col in pivot.columns if col != "Dataset"]

    wins = pd.Series(0, index=cols_to_highlight, dtype=float)

    for idx, row in pivot.loc[pivot.index != "Avg Rank", cols_to_highlight].iterrows():
        max_val = row.min() if minimise else row.max()
        winners = row[row == max_val].index
        k = len(winners)

        frac = 1.0 / k if k > 0 else 0.0
        wins[winners] += frac

    wins_row = pd.DataFrame([wins], index=["Wins"])
    wins_row["Dataset"] = "Wins"
    pivot_with_wins = pd.concat([pivot, wins_row], axis=0)
    pivot_with_wins = sort_dataset(pivot_with_wins)

    pivot_with_wins.style.apply(highlight, subset=cols_to_highlight, axis=1).format("{:.3f}").to_excel(
        f"{path}.xlsx", engine="openpyxl", index=False)
    csv = pd.read_excel(f"{path}.xlsx", engine="openpyxl")
    csv.to_csv(f"{path}.csv", index=False)
    save_statistical_significance(path, csv, avg_ranks.iloc[0])


def highlight_significant(val):
    return "background-color: green; color: white" if val <= 0.05 else ""


def save_nemenyi_test(path, data, avg_ranks):
    nemenyi_p_values = sp.posthoc_nemenyi_friedman(data)
    # fig = plt.figure(figsize=(10, 2), dpi=100)
    # sp.critical_difference_diagram(avg_ranks, nemenyi_p_values)
    # fig.savefig(f"{path}.png", dpi=300, bbox_inches="tight")
    # plt.show()
    CD, significant_differences = calculate_nemenyi_test(avg_ranks, 18)
    print(F"\nNemenyi's test (CD = {CD:.4f}):")
    with open(f'{path}-nemenyi.txt', 'w') as f:
        f.write(f"Nemenyi's test (CD = {CD:.2f}):\n")
        for sd in significant_differences:
            print(sd)
            f.write(f"{sd}\n")

    nemenyi_p_values.columns = data.columns
    nemenyi_p_values.index = data.columns
    nemenyi_p_values.round(3).to_csv(f"{path}-nemenyi.csv")
    nemenyi_p_values.round(3).style.map(highlight_significant).to_excel(
        f"{path}-nemenyi.xlsx", engine="openpyxl", index=False)


def calculate_nemenyi_test(avg_ranks, N):
    q05 = {
        "2": 1.960,
        "3": 2.343,
        "4": 2.569,
        "5": 2.728,
        "6": 2.850,
        "7": 2.949,
        "8": 3.031,
        "9": 3.102,
        "10": 3.164
    }
    K = len(avg_ranks) - 1  # methods (need to subtract 1 because 'Dataset' column is also present)
    term = (K * (K + 1)) / (6 * N)
    sqrt = math.sqrt(term)
    CD = q05[f"{K}"] * sqrt
    significant_differences = []
    comparisons = list(combinations(avg_ranks.items(), 2))
    comparisons = [((name1, value1), (name2, value2)) for ((name1, value1), (name2, value2)) in comparisons if
                   name1 != "Dataset" and name2 != "Dataset"]
    for ((name1, value1), (name2, value2)) in comparisons:
        if name1 == name2:
            continue
        diff = abs(value1 - value2)
        significant = diff >= CD
        comparison = f"{name1} and {name2} -> |{round(value1, 4)} - {round(value2, 4)}| = {round(diff, 4)} -> {round(diff, 4)} {">=" if diff >= CD else "<"} {round(CD, 4)} ({'Significant' if significant else 'Not Significant'})"
        if significant:
            significant_differences.append(comparison)
    return CD, significant_differences


def save_holm_test(path, data, avg_ranks):
    control_name = avg_ranks.idxmin()
    control_rank = avg_ranks[control_name]
    significant_differences = calculate_holm_test(avg_ranks, 18, control_name, control_rank)
    print(F"\nHolm's test:")
    with open(f'{path}-holm.txt', 'w') as f:
        f.write(f"Holm's test (control: {control_name}):\n")
        for sd in significant_differences:
            print(sd)
            f.write(f"{sd}\n")


def calculate_holm_test(avg_ranks, N, control_name, control_rank):
    control = (control_name, control_rank)
    comparisons = [(control, (name, value)) for (name, value) in avg_ranks.items() if name != control_name]
    comparisons = [((name1, value1), (name2, value2)) for ((name1, value1), (name2, value2)) in comparisons if
                   name1 != "Dataset" and name2 != "Dataset"]
    K = len(avg_ranks) - 1  # methods (need to subtract 1 because 'Dataset' column is also present)
    term = (K * (K + 1)) / (6 * N)
    sqrt = math.sqrt(term)
    p_values = []
    for ((name1, value1), (name2, value2)) in comparisons:
        z = (value1 - value2) / sqrt
        p_value = 2 * (1 - stats.norm.cdf(abs(z)))
        p_values.append((name2, p_value))
    p_values.sort(key=lambda x: x[1])
    alpha = 0.05
    i = 1
    significant_differences = []
    for (name, p_value) in p_values:
        adj_alpha = alpha / (K - i)
        significant = p_value <= adj_alpha
        cmp = "<" if round(p_value, 4) <= round(adj_alpha, 4) else ">="
        comparison = f"{control_name} vs {name} -> p:{round(p_value, 4)} {cmp} adj_a:{round(adj_alpha, 4)} ({'Significant' if significant else 'Not Significant'})"
        significant_differences.append(comparison)
        i = i + 1
        if not significant:
            break
    return significant_differences


def save_statistical_significance(path, data, avg_ranks):
    ranks = pd.to_numeric(avg_ranks, errors="coerce")
    data.drop(columns=["Dataset"], inplace=True)
    data = data[:-2]  # get all rows except for Avg Ranks and Avg Wins
    stat, p = friedmanchisquare(*[data[col] for col in data.columns])
    significant = "(Significant)" if p <= 0.05 else "(Not Significant)"
    friedman_result = f"Friedman test: p-value = {p:.4f} {significant}"
    print(friedman_result)
    with open(f'{path}-friedman.txt', 'w') as f:
        f.write(friedman_result)
    if p <= 0.05:
        save_nemenyi_test(path, data, ranks)
        save_holm_test(path, data, ranks)

In [35]:
measures = ["Macro-AUROC", "Hamming Loss", "Subset Accuracy", "Macro-F1", "Macro-Precision", "Macro-Recall",
            "Ranking Loss"]
other_stats = ["Nodes; Leaves"]
for comp_name in approaches_list:
    approaches = approaches_list[comp_name]
    for measure in measures:
        save_measure_table(measure, comp_name)


-- Macro-AUROC all --
Friedman test: p-value = 0.0112 (Significant)

Nemenyi's test (CD = 2.1235):
XofN_none N/A and XofN VarRed Org -> |5.06 - 2.67| = 2.39 -> 2.39 >= 2.1235 (Significant)

Holm's test:
XofN VarRed Org vs XofN_none N/A -> p:0.0009 < adj_a:0.0083 (Significant)
XofN VarRed Org vs XofN Vouk Ung -> p:0.008 < adj_a:0.01 (Significant)
XofN VarRed Org vs XofN Vouk Org -> p:0.0098 < adj_a:0.0125 (Significant)
XofN VarRed Org vs XofN Jac-max Ung -> p:0.0314 >= adj_a:0.0167 (Not Significant)

-- Hamming Loss all --
Friedman test: p-value = 0.0103 (Significant)

Nemenyi's test (CD = 2.1235):
XofN_none N/A and XofN Vouk Org -> |3.14 - 5.44| = 2.3 -> 2.3 >= 2.1235 (Significant)
XofN Jac-max Ung and XofN Vouk Org -> |2.92 - 5.44| = 2.52 -> 2.52 >= 2.1235 (Significant)

Holm's test:
XofN Jac-max Ung vs XofN Vouk Org -> p:0.0005 < adj_a:0.0083 (Significant)
XofN Jac-max Ung vs XofN Vouk Ung -> p:0.0314 >= adj_a:0.01 (Not Significant)

-- Subset Accuracy all --
Friedman test: p-value 

In [13]:
# reset approaches to all
approaches = [
    "Dataset",  # index column
    "XofN_none N/A",
    "XofN Jac-max Ung",
    "XofN Jac-max Org",
    "XofN VarRed Ung",
    "XofN VarRed Org",
    "XofN Vouk Ung",
    "XofN Vouk Org",
]

In [14]:
# Nodes; Leaves
all_results_table = pd.read_csv("all_results_table.csv")
all_results_table.loc[
    all_results_table["XofN method"] == "XofN_none", "Ung or Org"
] = "N/A"
all_results_table.columns = all_results_table.columns.str.strip()
pivot = all_results_table.pivot_table(
    index=["Dataset"],
    columns=["XofN method", "Ung or Org"],
    values="Nodes; Leaves",
    aggfunc="first"
)

pivot.columns = [f"{a} {b}" for a, b in pivot.columns]
pivot = pivot.reset_index()
pivot = pivot.sort_values(by="Dataset", ascending=False)
pivot_column_order = approaches.copy()
pivot = pivot.reindex(columns=pivot_column_order)
pivot = sort_dataset(pivot)
pivot.to_csv("Nodes Leaves.csv", index=False)
pivot

,Dataset,XofN_none N/A,XofN Jac-max Ung,XofN Jac-max Org,XofN VarRed Ung,XofN VarRed Org,XofN Vouk Ung,XofN Vouk Org
13,fingerprint_high_freq,18.8; 9.9,43.8; 22.4,46.4; 23.7,42.4; 21.7,48.0; 24.5,35.8; 18.4,45.0; 23.0
15,fingerprint_mid_freq,161.0; 81.0,158.0; 79.5,164.0; 82.5,145.2; 73.1,152.2; 76.6,133.6; 67.3,172.8; 86.9
14,fingerprint_low_freq,8.8; 4.9,1.0; 1.0,7.6; 4.3,27.6; 14.3,27.2; 14.1,50.0; 25.5,63.0; 32.0
12,fingerprint_cardiac,8.6; 4.8,18.6; 9.8,35.4; 18.2,52.2; 26.6,55.8; 28.4,46.6; 23.8,54.4; 27.7
16,fingerprint_ten,151.6; 76.3,160.6; 80.8,154.4; 77.7,149.4; 75.2,158.4; 79.7,136.4; 68.7,163.4; 82.2
17,fingerprint_ten_mid,85.6; 43.3,98.4; 49.7,109.6; 55.3,89.4; 45.2,95.6; 48.3,80.4; 40.7,102.8; 51.9
7,CPI_high_freq,22.4; 11.7,17.0; 9.0,29.6; 15.3,28.6; 14.8,33.8; 17.4,14.2; 7.6,32.0; 16.5
9,CPI_mid_freq,127.8; 64.4,108.8; 54.9,132.8; 66.9,95.0; 48.0,112.4; 56.7,77.8; 39.4,119.8; 60.4
8,CPI_low_freq,49.4; 25.2,53.2; 27.1,57.2; 29.1,42.4; 21.7,49.0; 25.0,53.8; 27.4,60.4; 30.7
6,CPI_cardiac,52.0; 26.5,55.4; 28.2,64.4; 32.7,46.0; 23.5,57.4; 29.2,46.2; 23.6,53.0; 27.0


In [15]:
# #XofN; #Feat/XofN
all_results_table = pd.read_csv("all_results_table.csv")
all_results_table.loc[
    all_results_table["XofN method"] == "XofN_none", "Ung or Org"
] = "N/A"
all_results_table.columns = all_results_table.columns.str.strip()
pivot = all_results_table.pivot_table(
    index=["Dataset"],
    columns=["XofN method", "Ung or Org"],
    values="#XofN; #Feat/XofN",
    aggfunc="first"
)

pivot.columns = [f"{a} {b}" for a, b in pivot.columns]
pivot = pivot.reset_index()
pivot = pivot.sort_values(by="Dataset", ascending=False)
pivot_column_order = approaches.copy()
pivot = pivot.reindex(columns=pivot_column_order)
pivot = sort_dataset(pivot)
pivot.to_csv("#XofN #Feat per XofN.csv", index=False)
pivot

,Dataset,XofN_none N/A,XofN Jac-max Ung,XofN Jac-max Org,XofN VarRed Ung,XofN VarRed Org,XofN Vouk Ung,XofN Vouk Org
13,fingerprint_high_freq,NaN,108.0; 5.0,108.0; 5.0,110.9; 4.9,110.9; 4.9,78.1; 3.1,78.1; 3.1
15,fingerprint_mid_freq,NaN,108.0; 5.0,108.0; 5.0,110.1; 4.9,110.1; 4.9,78.7; 2.8,78.7; 2.8
14,fingerprint_low_freq,NaN,108.0; 5.0,108.0; 5.0,112.0; 4.8,112.0; 4.8,141.9; 3.5,141.9; 3.5
12,fingerprint_cardiac,NaN,108.0; 5.0,108.0; 5.0,111.0; 4.8,111.0; 4.8,78.0; 3.7,78.0; 3.7
16,fingerprint_ten,NaN,108.0; 5.0,108.0; 5.0,109.8; 4.9,109.8; 4.9,108.9; 2.3,108.9; 2.3
17,fingerprint_ten_mid,NaN,108.0; 5.0,108.0; 5.0,110.8; 4.9,110.8; 4.9,152.9; 2.9,152.9; 2.9
7,CPI_high_freq,NaN,322.0; 5.0,322.0; 5.0,343.6; 4.6,343.6; 4.6,87.9; 3.2,87.9; 3.2
9,CPI_mid_freq,NaN,322.0; 5.0,322.0; 5.0,335.6; 4.7,335.6; 4.7,140.2; 2.3,140.2; 2.3
8,CPI_low_freq,NaN,322.0; 5.0,322.0; 5.0,331.2; 4.8,331.2; 4.8,393.1; 2.9,393.1; 2.9
6,CPI_cardiac,NaN,322.0; 5.0,322.0; 5.0,355.6; 4.4,355.6; 4.4,385.2; 2.8,385.2; 2.8


In [16]:
# #Ung. Feats
all_results_table = pd.read_csv("all_results_table.csv")
all_results_table.loc[
    all_results_table["XofN method"] == "XofN_none", "Ung or Org"
] = "N/A"
all_results_table.columns = all_results_table.columns.str.strip()
all_results_table["#Ung. Feats"] = all_results_table["#Ung. Feats"].round(2)
pivot = all_results_table.pivot_table(
    index=["Dataset"],
    columns=["XofN method", "Ung or Org"],
    values="#Ung. Feats",
    aggfunc="first"
)

pivot.columns = [f"{a} {b}" for a, b in pivot.columns]
pivot = pivot.reset_index()
pivot = pivot.sort_values(by="Dataset", ascending=False)
pivot_column_order = approaches.copy()
pivot = pivot.reindex(columns=pivot_column_order)
pivot = sort_dataset(pivot)
pivot.to_csv("#Ung. Feats.csv", index=False)
pivot

,Dataset,XofN_none N/A,XofN Jac-max Ung,XofN Jac-max Org,XofN VarRed Ung,XofN VarRed Org,XofN Vouk Ung,XofN Vouk Org
13,fingerprint_high_freq,NaN,0.0,0.0,2.05,2.05,469.8,469.8
15,fingerprint_mid_freq,NaN,0.0,0.0,1.57,1.57,461.2,461.2
14,fingerprint_low_freq,NaN,0.0,0.0,0.92,0.92,413.1,413.1
12,fingerprint_cardiac,NaN,0.0,0.0,2.57,2.57,446.2,446.2
16,fingerprint_ten,NaN,0.0,0.0,0.96,0.96,449.0,449.0
17,fingerprint_ten_mid,NaN,0.0,0.0,1.48,1.48,408.1,408.1
7,CPI_high_freq,NaN,0.0,0.0,9.27,9.27,1551.6,1551.6
9,CPI_mid_freq,NaN,0.0,0.0,33.04,33.04,1528.5,1528.5
8,CPI_low_freq,NaN,0.0,0.0,2.06,2.06,1456.7,1456.7
6,CPI_cardiac,NaN,0.0,0.0,25.00,25.00,1476.7,1476.7


In [17]:
# XofN time (s); PCT tr. time (s)
all_results_table = pd.read_csv("all_results_table.csv")
all_results_table.loc[
    all_results_table["XofN method"] == "XofN_none", "Ung or Org"
] = "N/A"
all_results_table.columns = all_results_table.columns.str.strip()
pivot = all_results_table.pivot_table(
    index=["Dataset"],
    columns=["XofN method", "Ung or Org"],
    values="XofN time (s); PCT tr. time (s)",
    aggfunc="first"
)

pivot.columns = [f"{a} {b}" for a, b in pivot.columns]
pivot = pivot.reset_index()
pivot = pivot.sort_values(by="Dataset", ascending=False)
pivot_column_order = approaches.copy()
pivot = pivot.reindex(columns=pivot_column_order)
pivot = sort_dataset(pivot)
pivot.to_csv("XofN time (s) PCT tr. time (s).csv", index=False)
pivot

,Dataset,XofN_none N/A,XofN Jac-max Ung,XofN Jac-max Org,XofN VarRed Ung,XofN VarRed Org,XofN Vouk Ung,XofN Vouk Org
13,fingerprint_high_freq,1.0,78.2; 0.7,78.2; 1.3,94.5; 0.7,94.5; 1.3,4.1; 0.6,4.1; 1.4
15,fingerprint_mid_freq,1.1,78.3; 0.7,78.3; 1.3,94.7; 0.7,94.7; 1.3,4.3; 0.6,4.3; 1.2
14,fingerprint_low_freq,1.1,78.4; 0.7,78.4; 1.3,94.7; 0.7,94.7; 1.3,6.7; 0.8,6.7; 1.5
12,fingerprint_cardiac,1.1,78.1; 0.7,78.1; 1.3,94.1; 0.7,94.1; 1.3,5.7; 0.7,5.7; 1.4
16,fingerprint_ten,1.1,74.4; 0.7,74.4; 1.3,94.2; 0.7,94.2; 1.4,3.9; 0.7,3.9; 1.4
17,fingerprint_ten_mid,1.1,75.7; 0.7,75.7; 1.3,96.1; 0.7,96.1; 1.3,4.5; 0.8,4.5; 1.4
7,CPI_high_freq,2.4,356.2; 0.9,356.2; 2.6,466.8; 0.9,466.8; 2.6,16.3; 0.6,16.3; 2.7
9,CPI_mid_freq,2.5,359.1; 1.0,359.1; 2.9,463.5; 0.9,463.5; 2.5,15.8; 0.8,15.8; 2.6
8,CPI_low_freq,2.5,358.1; 1.1,358.1; 3.0,459.8; 0.9,459.8; 2.6,22.9; 1.2,22.9; 3.2
6,CPI_cardiac,2.8,355.0; 1.0,355.0; 3.0,464.8; 0.9,464.8; 2.6,26.3; 1.2,26.3; 3.4
